# 🔧 Lesson 05: Tool Use — Function Calling & External APIs

Agents become truly powerful when they can USE tools. This lesson covers OpenAI function calling and building a real tool library for your DSA agent.

In [ ]:
import os, json, subprocess, requests
from openai import AzureOpenAI
from dotenv import load_dotenv
from rich.console import Console
load_dotenv()
client = AzureOpenAI(
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION')
)
DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o')
console = Console()
console.print('[green]✅ Ready![/green]')

## 🛠️ Part 1: Define Tools (OpenAI Function Schema)

In [ ]:
# Tools are defined as JSON schemas for the LLM to understand
TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'run_python_code',
            'description': 'Execute Python code and return the output. Use for testing solutions.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'code': {'type': 'string', 'description': 'Python code to execute'}
                },
                'required': ['code']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_dsa_info',
            'description': 'Get time/space complexity and key info about a DSA algorithm or data structure.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'topic': {'type': 'string', 'description': 'DSA topic e.g. merge sort, binary search, hashmap'}
                },
                'required': ['topic']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'search_leetcode_problems',
            'description': 'Find relevant LeetCode problems for a given topic or pattern.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'topic': {'type': 'string'},
                    'difficulty': {'type': 'string', 'enum': ['easy', 'medium', 'hard', 'any']}
                },
                'required': ['topic']
            }
        }
    }
]

# --- Tool implementations ---
def run_python_code(code: str) -> str:
    result = subprocess.run(['python', '-c', code], capture_output=True, text=True, timeout=10)
    return result.stdout or result.stderr

def get_dsa_info(topic: str) -> str:
    db = {
        'merge sort': 'Time: O(n log n) | Space: O(n) | Stable sort, good for linked lists',
        'quick sort': 'Time: O(n log n) avg, O(n²) worst | Space: O(log n) | In-place, fast in practice',
        'binary search': 'Time: O(log n) | Space: O(1) | Requires sorted array',
        'hashmap': 'Time: O(1) avg | Space: O(n) | Best for fast lookup/frequency counting',
        'bfs': 'Time: O(V+E) | Space: O(V) | Level-order traversal, shortest path in unweighted graph',
        'dfs': 'Time: O(V+E) | Space: O(V) | Explores deep paths, used in cycle detection',
        'dynamic programming': 'Time: varies | Overlapping subproblems + optimal substructure pattern',
    }
    return db.get(topic.lower(), f'No info for {topic}. Common topics: merge sort, binary search, hashmap, bfs, dfs, dp')

def search_leetcode_problems(topic: str, difficulty: str = 'any') -> str:
    problems = {
        'array': ['Two Sum (Easy) #1', 'Best Time to Buy Stock (Easy) #121', 'Maximum Subarray (Medium) #53'],
        'stack': ['Valid Parentheses (Easy) #20', 'Min Stack (Medium) #155', 'Daily Temperatures (Medium) #739'],
        'binary search': ['Binary Search (Easy) #704', 'Search in Rotated Array (Medium) #33', 'Find Peak Element (Medium) #162'],
        'dynamic programming': ['Climbing Stairs (Easy) #70', 'Coin Change (Medium) #322', 'Longest Increasing Subsequence (Medium) #300'],
        'graph': ['Number of Islands (Medium) #200', 'Course Schedule (Medium) #207', 'Clone Graph (Medium) #133'],
    }
    result = problems.get(topic.lower(), ['No specific problems found. Try: array, stack, binary search, dp, graph'])
    return json.dumps(result)

TOOL_MAP = {
    'run_python_code': run_python_code,
    'get_dsa_info': get_dsa_info,
    'search_leetcode_problems': search_leetcode_problems
}

console.print('[green]✅ 3 tools registered![/green]')

## 🤖 Part 2: Agent with Tool Calling Loop

In [ ]:
def run_agent_with_tools(user_question: str, max_rounds: int = 5):
    messages = [
        {'role': 'system', 'content': 'You are a DSA expert. Use tools to provide accurate information and test code.'},
        {'role': 'user', 'content': user_question}
    ]
    
    console.print(f'[cyan]User:[/cyan] {user_question}')
    
    for round_num in range(max_rounds):
        response = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            tools=TOOLS,
            tool_choice='auto',
            temperature=0
        )
        
        msg = response.choices[0].message
        
        # No tool calls — we have final answer
        if not msg.tool_calls:
            console.print(f'\n[bold green]Answer:[/bold green] {msg.content}')
            return msg.content
        
        # Process tool calls
        messages.append({'role': 'assistant', 'tool_calls': [
            {'id': tc.id, 'type': 'function', 'function': {'name': tc.function.name, 'arguments': tc.function.arguments}}
            for tc in msg.tool_calls
        ]})
        
        for tc in msg.tool_calls:
            tool_name = tc.function.name
            tool_args = json.loads(tc.function.arguments)
            
            console.print(f'[yellow]🔧 Tool:[/yellow] {tool_name}({tool_args})')
            result = TOOL_MAP[tool_name](**tool_args)
            console.print(f'[magenta]↳ Result:[/magenta] {result[:100]}')
            
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': str(result)})
    
    return 'Max rounds reached'


# Test with different questions
run_agent_with_tools('What is the time complexity of merge sort? Also give me 3 LeetCode problems on sorting.')

In [ ]:
# Test code execution tool
run_agent_with_tools('Write and run a Python function to check if a string is a palindrome. Test it with "racecar" and "hello".')